**Портфельная стратегия на основе модели Гармана-Колхагена**

В данном проекте нами будет реализована оценка стоимости опционов на основе данных по доходностям государственных облигаций США и Германии, ожидаемой волатильности, а также курса валют (EUR/USD) с использованием модели Гармана-Колхагена оценки стоимости валютных опционов.

Цель состоит в том, чтобы реализовать оптимальный портфель, состоящий из купленных опционов колл европейского типа со страйком 90%, продажу опционов колл со страйком 110%, а также покупки или продажи иностранной валюты. Все опционы сроком на 1 год. Портфель должен быть дельта и гамма нейтральным. Требуется описать построение портфеля и проанализировать полученные результаты.

Сначала определимся с данными, которые мы будем использовать. Рассмотрим модель Гармана-Колхагена оценки стоимости опционов колл европейского типа:

Модель Гармана-Колхагена для европейского опциона колл на иностранную валюту:

$$
C = S_0 e^{-r_f T} N(d_1) - K e^{-r_d T} N(d_2)
$$

где:

$$
d_1 = \frac{\ln(S_0/E) + \left(r_d - r_f + \frac{\sigma^2}{2}\right)T}{\sigma \sqrt{T}}
$$

$$
d_2 = d_1 - \sigma \sqrt{T}
$$

Обозначения:

- $S_0$ — текущий спотовый курс иностранной валюты;
- $K$ — цена исполнения (страйк);
- $r_d$ — безрисковая процентная ставка в домашней валюте;
- $r_f$ — безрисковая процентная ставка в иностранной валюте;
- $T$ — время до истечения опциона (в годах);
- $\sigma$ — волатильность обменного курса;
- $N(\cdot)$ — функция стандартного нормального распределения.

Таким образом, нам необходимо знать безрисковые процентные ставки, подразумеваемую волатильность, текущий спотовый курс.

В качестве безрисковых процентных ставок в долларах США мы будем брать ставку по государственным облигациям США сроком на один год (US 1-year Treasury). Данные доступны на сайте CNBC и на момент написания этих строк были 4,41%.

В случае безрисковой ставки для EUR ситуация несколько сложнее, так как здесь нет единого эмитента облигаций. Поэтому мы будем использовать облигации государств с рейтингом AAA (таких, как Германия), однако для этих же целей можно использовать и взвешенную сумму процентных ставок государств с рейтингом AAA. В нашем случае мы рассматриваем облигации German 1-year Bund Yield, данные доступны на сайте cbonds.com и на момент написания этих строк были 2,90%.

Курс валют можно взять из данных Европейского центрального банка (ECB), на момент написания этих строк курс был 1,1193 USD за 1 EUR.

Пождразумеваемую волатильность берём из данных сервиса OpenBB, который можно установить с использованием pip или аналогичных средств.

In [ ]:
pip install openbb

In [2]:
from openbb import obb

# по умолчанию EUR/USD
result = obb.cftc.fx_implied_vol(provider='cftc')

print(f'Подразумеваемая волатильность на срок 1 год: {result.results[0].vol_1y}')


Extensions to add: bls@2.0.0, cboe@2.0.0, cftc@2.0.0, congress_gov@2.0.0, deribit@2.0.0, ecb@2.0.0, eia@2.0.0, famafrench@2.0.0, federal_reserve@2.0.0, finra@2.0.0, fred@2.0.0, imf@3.0.0, jodi@2.0.0, nasdaq@2.0.0, news@2.0.0, oecd@2.0.0, openbb_charting@4.0.0, sec@2.0.0, tmx@2.0.0, us_eia@2.0.0, us_treasury@2.0.0, uscongress@2.0.0, usda@2.0.0, ustreasury@2.0.0

Building...
Подразумеваемая волатильность на срок 1 год: 8.572351494679012


Таким образом, оценки, полученные с помощью средств OpenBB, на момент написания этих строк были 8,57%.

Теперь у нас есть все средства для оценки стоимости опционов колл на иностранную валюту.

In [3]:
import numpy as np
from scipy.stats import norm

# параметры
S_0 = 1.1193
r_d = 0.0441
r_f = 0.0290
sigma = 0.0857
E1 = 0.9 * S_0
E2 = 1.1 * S_0  # для второго опциона
T = 1

d_1 = (np.log(S_0 / E1) + (r_d - r_f + sigma**2 / 2) * T) / (sigma * T**0.5)
d_2 = d_1 - sigma * T**0.5
D_1 = (np.log(S_0 / E2) + (r_d - r_f + sigma**2 / 2) * T) / (sigma * T**0.5)
D_2 = D_1 - sigma * T**0.5

C1 = S_0 * np.exp(-r_f * T) * norm.cdf(d_1) - E1 * np.exp(-r_d * T) * norm.cdf(d_2)
C2 = S_0 * np.exp(-r_f * T) * norm.cdf(D_1) - E2 * np.exp(-r_d * T) * norm.cdf(D_2)
print(f'Цена опциона первого типа: {C1:.2f}')
print(f'Цена опциона второго типа: {C2:.2f}')


Цена опциона первого типа: 0.13
Цена опциона второго типа: 0.01


Следующим этапом будет формирование дельта и гамма нейтральной стратегии. Дельта характеризует изменение цены инструмента с изменением цены базового актива. Гамма характеризует изменение дельты с изменением цены базового актива. Дельта-нейтральность фиксирует стоимость портфеля при незначительном изменении цены базового актива, а гамма-нейтральность обеспечивает фиксацию дельты (в нашем случае на нулевом значении) для незначительного изменения стоимости базового актива. Таким образом, полученный портфель будет либо продавать выпуклость (положительная тета, нам платят за то, что мы берём риск, связанный с возможной волатильностью), либо платить за стабильность (отрицательная тета, мы платим за стабильность и снижение риска, но можем заработать на значительной волатильности).

Для того, чтобы обеспечить дельта- и гамма-нейтральность портфеля, нам нужно составить систему уравнений для весов $w_1$, $w_2$, $w_3$ соответственно купленных опционов первого типа, проданных опционов второго типа и купленой или проданной валюты. Ясно, что тогда $w_1$ + $w_2$ + $w_3$ = 1.

Дельта портфеля есть взвешенная сумма дельт активов, входящих в его состав. Делта для опционов - производная по стоимости базового актива от стоимости опциона, гамма - соответственно вторая производная:

Греки опциона колл в модели Гармана-Колхагена:

**Дельта (Delta):**

$$
\Delta = \frac{\partial C}{\partial S_0} = e^{-r_f T} N(d_1)
$$

**Гамма (Gamma):**

$$
\Gamma = \frac{\partial^2 C}{\partial S_0^2} = \frac{e^{-r_f T} \, n(d_1)}{S_0 \sigma \sqrt{T}}
$$

где:

$$
n(d_1) = \frac{1}{\sqrt{2\pi}} e^{-\frac{d_1^2}{2}}
$$

— функция плотности стандартного нормального распределения, а $d_1$ определяется как:

$$
d_1 = \frac{\ln(S_0/K) + \left(r_d - r_f + \frac{\sigma^2}{2}\right)T}{\sigma \sqrt{T}}
$$

Дельта самой валюты есть 1 как производная по её же цене, а гамма соответственно 0.

Составим систему уравнений с учётом того, что мы продаём опционы второго типа.

Система уравнений для поиска весов портфеля $w_1$, $w_2$, $w_3$ (в долях от капитала $V$):

$$
\begin{cases}
w_1 + w_2 + w_3 = 1 \\[6pt]
w_1 \Delta_1 - w_2 \Delta_2 + w_3 = 0 \\[6pt]
w_1 \Gamma_1 - w_2 \Gamma_2 = 0
\end{cases}
$$

Решение этой системы произведём с помощью средств Python.

Далее, рассчитаем $\theta$ портфеля. Для этого необходимо взять производную от стоимости опциона по времени. Однако, формула для стоимости записана в терминах срока до конца действия опциона, а значит в итоговом результате знак будет противоположный:

$$
\Theta = \frac{\partial C}{\partial t} = -\frac{S_0 e^{-r_f T} n(d_1) \sigma}{2\sqrt{T}} + r_f S_0 e^{-r_f T} N(d_1) - r_d K e^{-r_d T} N(d_2)
$$

Тета всего портфеля есть количество первого актива, умноженное на его тету, минус количество вторго (проданного) актива, умноженное на тету. Вклад спотового курса ноль, так как у него нет временного распада.

Результаты расчётов приведены ниже.

In [6]:
import numpy as np
from scipy.stats import norm

# параметры
S_0 = 1.1193
r_d = 0.0441
r_f = 0.0290
sigma = 0.0857
T = 1
V = 100000  # стоимость портфеля

E1 = 0.9 * S_0
E2 = 1.1 * S_0  # для второго опциона

# для второго
D_1 = (np.log(S_0 / E2) + (r_d - r_f + sigma**2 / 2) * T) / (sigma * T**0.5)
D_2 = D_1 - sigma * T**0.5
# для первого
d_1 = (np.log(S_0 / E1) + (r_d - r_f + sigma**2 / 2) * T) / (sigma * T**0.5)
d_2 = d_1 - sigma * T**0.5

# цена опционов
C1 = S_0 * np.exp(-r_f * T) * norm.cdf(d_1) - E1 * np.exp(-r_d * T) * norm.cdf(d_2)
C2 = S_0 * np.exp(-r_f * T) * norm.cdf(D_1) - E2 * np.exp(-r_d * T) * norm.cdf(D_2)

# дельты и гаммы
delta_1 = np.exp(-r_f * T) * norm.cdf(d_1)
Gamma_1 = np.exp(-r_f * T) * norm.pdf(d_1) / (S_0 * sigma * T**0.5)
delta_2 = np.exp(-r_f * T) * norm.cdf(D_1)
Gamma_2 = np.exp(-r_f * T) * norm.pdf(D_1) / (S_0 * sigma * T**0.5)

# параметры портфеля (находим веса)
A = np.array([[delta_1, -delta_2, 1.0],
              [Gamma_1, -Gamma_2, 0.0],
              [1.0, 1.0, 1.0]])
b = np.array([0.0, 0.0, 1.0])
w1, w2, w3 = np.linalg.solve(A, b)

print(f'Веса: {w1:.2f}, {w2:.2f}, {w3:.2f}')

N1, N2, N3 = w1 * V / C1, w2 * V / C2, w3 * V / S_0

print(f'Покупаем {int(N1)} опционов I типа',
      f'\nПродаём {int(N2)} опционов II типа',
      f'\nПокупаем {int(N3)} EUR\n')

theta_1 = (
    -S_0 * np.exp(-r_f * T) * norm.pdf(d_1) * sigma / (2 * np.sqrt(T))
    + S_0 * r_f * np.exp(-r_f * T) * norm.cdf(d_1)
    - E1 * r_d * np.exp(-r_d * T) * norm.cdf(d_2)
)
theta_2 = (
    -S_0 * np.exp(-r_f * T) * norm.pdf(D_1) * sigma / (2 * np.sqrt(T))
    + S_0 * r_f * np.exp(-r_f * T) * norm.cdf(D_1)
    - E2 * r_d * np.exp(-r_d * T) * norm.cdf(D_2)
)

# тета портфеля на единицу актива
theta = N1 * theta_1 - N2 * theta_2

print(f'Тета портфеля (годовая): {theta:.2f} USD')
print(f'Абсолютная дневная тета: {theta / 365:.2f} USD')

Веса: 1.40, 0.73, -1.12
Покупаем 1102806 опционов I типа 
Продаём 7996305 опционов II типа 
Покупаем -100453 EUR

Тета портфеля (годовая): 103145.99 USD
Абсолютная дневная тета: 282.59 USD


Мы видим, что для реализации дельта и гамма нейтральной стратегии мы должны занять средства для покупки и продажи иностранного валюты (отрицательный вес) с весом 1.12, продать опционов второго типа с весом 0.73, а также купить на полученные средства опционов первого типа с весом 1.40 от исходного капитала. Годовая тета портфеля оказалась положительной (нам платят за риск, или мы продаём выпуклость), размер дохода при совпадении ожидаемой волатильности с реализованной будет 103145.16 USD, в дневном эквиваленте 282.95 USD. Таким образом, в дельта и гамма нейтральной стратегии мы получаем премию за риск при данных условиях, но теряем на волатильности.